In [4]:
import json
import re
import pandas as pd
import numpy as np

In [5]:
MAX_LEN = 256
PAD_IDX = 0
UNK_IDX = 1

print("Max URL length:", MAX_LEN)
print("PAD index:", PAD_IDX)
print("UNK index:", UNK_IDX)

Max URL length: 256
PAD index: 0
UNK index: 1


In [6]:
with open("../data/vocab.json", "r") as f:
    vocab_data = json.load(f)

index_to_string = vocab_data["index_to_string"]

char_to_index = {
    char: index
    for index, char in enumerate(index_to_string)
}

print("Vocabulary size:", len(index_to_string))
print("PAD index:", char_to_index["<PAD>"])
print("UNK index:", char_to_index["<UNK>"])
print("First 10 entries:", index_to_string[:10])

Vocabulary size: 67
PAD index: 0
UNK index: 1
First 10 entries: ['<PAD>', '<UNK>', '!', '#', '$', '%', '&', '(', ')', '*']


In [7]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Training set: (185397, 2)
Validation set: (23175, 2)
Test set: (23175, 2)

Training labels:
label
1    107845
0     77552
Name: count, dtype: int64

Validation labels:
label
1    13481
0     9694
Name: count, dtype: int64

Test labels:
label
1    13481
0     9694
Name: count, dtype: int64


In [8]:
def canonicalize_model_url(url):
    url = str(url).strip()

    # Remove URI scheme
    url = re.sub(
        r"^[A-Za-z][A-Za-z0-9+.\-]*://",
        "",
        url
    )

    # Remove leading www/www2/www3
    url = re.sub(
        r"^www\d*\.",
        "",
        url,
        flags=re.IGNORECASE
    )

    # Remove trailing slash
    url = re.sub(r"/+$", "", url)

    return url

test_url = "https://www.example.com/test/"

print("Before:", test_url)
print("After: ", canonicalize_model_url(test_url))

Before: https://www.example.com/test/
After:  example.com/test


In [9]:
train_df["URL"] = train_df["URL"].apply(canonicalize_model_url)
val_df["URL"] = val_df["URL"].apply(canonicalize_model_url)
test_df["URL"] = test_df["URL"].apply(canonicalize_model_url)

print(train_df.head())

                      URL  label
0    southbankmosaics.com      1
1            uni-mainz.de      1
2         sfnmjournal.com      1
3  rewildingargentina.org      1
4     globalreporting.org      1


In [10]:
def encode_url(url):
    indices = []

    for char in url:
        if char in char_to_index:
            indices.append(char_to_index[char])
        else:
            indices.append(UNK_IDX)

    # Truncate if longer than MAX_LEN
    indices = indices[:MAX_LEN]

    # Pad if shorter than MAX_LEN
    while len(indices) < MAX_LEN:
        indices.append(PAD_IDX)

    return indices

test_encoded = encode_url(train_df["URL"].iloc[0])

print("URL:", train_df["URL"].iloc[0])
print("Encoded length:", len(test_encoded))
print("First 20 indices:", test_encoded[:20])

URL: southbankmosaics.com
Encoded length: 256
First 20 indices: [51, 47, 53, 52, 40, 34, 33, 46, 43, 45, 47, 51, 33, 41, 35, 51, 13, 35, 47, 45]


In [11]:
short_url = "abc.com"
encoded_short = encode_url(short_url)

print("Short URL length:", len(short_url))
print("Encoded length:", len(encoded_short))
print("Last 10 indices:", encoded_short[-10:])

unknown_url = "abc😀.com"
encoded_unknown = encode_url(unknown_url)

print("\nUnknown character test:")
print("URL:", unknown_url)
print("Encoded:", encoded_unknown[:10])

Short URL length: 7
Encoded length: 256
Last 10 indices: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Unknown character test:
URL: abc😀.com
Encoded: [33, 34, 35, 1, 13, 35, 47, 45, 0, 0]


In [12]:
def encode_dataframe(df):
    X = [encode_url(url) for url in df["URL"]]
    y = df["label"].tolist()

    return X, y

X_train, y_train = encode_dataframe(train_df)
X_val, y_val = encode_dataframe(val_df)
X_test, y_test = encode_dataframe(test_df)

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Test examples:", len(X_test))

print("Length of first training URL:", len(X_train[0]))
print("First training label:", y_train[0])

Training examples: 185397
Validation examples: 23175
Test examples: 23175
Length of first training URL: 256
First training label: 1


In [13]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.7.0+cu128
CUDA available: True
GPU: Tesla T4


In [14]:
from torch.utils.data import Dataset


class URLDataset(Dataset):
    def __init__(self, df):
        self.urls = df["URL"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, index):
        encoded_url = encode_url(self.urls[index])

        # Dataset uses:
        # 0 = phishing
        # 1 = legitimate
        # For training, we will later convert this so:
        # 1 = phishing
        label = 1 if self.labels[index] == 0 else 0

        return torch.tensor(encoded_url, dtype=torch.long), torch.tensor(label, dtype=torch.float32)

In [15]:
train_dataset = URLDataset(train_df)

print("Number of training examples:", len(train_dataset))

url_tensor, label = train_dataset[0]

print("URL tensor shape:", url_tensor.shape)
print("Label:", label.item())

Number of training examples: 185397
URL tensor shape: torch.Size([256])
Label: 0.0


In [16]:
from torch.utils.data import DataLoader

train_dataset = URLDataset(train_df)
val_dataset = URLDataset(val_df)
test_dataset = URLDataset(test_df)

BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 725
Validation batches: 91
Test batches: 91


In [17]:
X_batch, y_batch = next(iter(train_loader))

print("Input batch shape:", X_batch.shape)
print("Label batch shape:", y_batch.shape)
print("First 10 labels:", y_batch[:10])

Input batch shape: torch.Size([256, 256])
Label batch shape: torch.Size([256])
First 10 labels: tensor([0., 0., 0., 0., 0., 0., 1., 0., 1., 0.])


In [18]:
import torch.nn as nn


class CharacterCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.conv1 = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=128,
            kernel_size=5
        )

        self.conv2 = nn.Conv1d(
            in_channels=128,
            out_channels=128,
            kernel_size=5
        )

        self.relu = nn.ReLU()

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.embedding(x)

        # Change from:
        # (batch, sequence, embedding)
        # to:
        # (batch, embedding, sequence)
        x = x.transpose(1, 2)

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))

        x = self.pool(x)

        # Remove the final dimension
        x = x.squeeze(2)

        x = self.fc(x)

        return x

In [19]:
VOCAB_SIZE = len(index_to_string)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CharacterCNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=32
).to(device)

print("Device:", device)
print("Model created successfully!")

Device: cuda
Model created successfully!


In [20]:
X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

outputs = model(X_batch)

print("Input shape:", X_batch.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([256, 256])
Output shape: torch.Size([256, 1])


In [21]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [22]:
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):

    # Training
    model.train()

    total_train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch).squeeze(1)

        # Calculate loss
        loss = criterion(outputs, y_batch)

        # Backward pass
        loss.backward()

        # Update model weights
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()

    total_val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch).squeeze(1)

            loss = criterion(outputs, y_batch)

            total_val_loss += loss.item()

    average_val_loss = total_val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} "
        f"- Train Loss: {average_train_loss:.4f} "
        f"- Val Loss: {average_val_loss:.4f}"
    )

Epoch 1/5 - Train Loss: 0.3056 - Val Loss: 0.2795
Epoch 2/5 - Train Loss: 0.2700 - Val Loss: 0.2662
Epoch 3/5 - Train Loss: 0.2578 - Val Loss: 0.2750
Epoch 4/5 - Train Loss: 0.2492 - Val Loss: 0.2608
Epoch 5/5 - Train Loss: 0.2399 - Val Loss: 0.2581


In [23]:
torch.save(
    model.state_dict(),
    "../models/cnn_baseline.pt"
)

print("Model saved successfully.")

Model saved successfully.


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model.eval()

all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch).squeeze(1)

        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities >= 0.5).float()

        all_probabilities.extend(probabilities.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(y_batch.numpy())

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)
cm = confusion_matrix(all_labels, all_predictions)

print("MODEL 1 - Test Set Performance")
print("--------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

MODEL 1 - Test Set Performance
--------------------------------
Accuracy:  0.9061
Precision: 0.9601
Recall:    0.8093
F1 Score:  0.8783

Confusion Matrix:
[[13155   326]
 [ 1849  7845]]


In [25]:
baseline_results = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

print(baseline_results)

{'accuracy': 0.9061488673139159, 'precision': 0.9601028025945417, 'recall': 0.8092634619352177, 'f1': 0.8782535684298909}
